# 06 — Prospective registration (Path C): out-of-sample novel-indication lock

Reproduces the **locked, outcome-blind prospective prediction set** committed before any
of these ongoing Phase III trials reports. The model and its training data are frozen; this
notebook only *selects, attributes and filters* the trials and *applies* the frozen model.

**Pipeline** (committed scripts in `scripts/benchmark/`):

| stage | script | output |
|---|---|---|
| pull ongoing P3 | `prereg_pull_ongoing_p3` | ct.gov trials of cohort compounds |
| select novel pairs | `prereg_C_select_pairs` | (drug, indication) absent from training |
| recompute mechanism | `prereg_C_build_mech` | target→disease mech per pair (topology/KEGG/OT/ClinGen/in-module + direct-target) |
| recompute genetics | `prereg_C_build_genetics` | Mendelian (ClinVar/OT-causal), DepMap lineage dependency, mechanism-impact per pair — **de-imputed** from cached biology |
| pull missing modules | `prereg_C_pull_missing_modules` | OT disease modules absent from the cohort caches (e.g. overactive bladder for neurogenic detrusor overactivity) |
| lock (fit-on-all) | `prereg_C_lock` | full set (1,470) + SHA |
| **attribution** | `prereg_C_finalize_clean` | drug = experimental **differentiator** |
| **novelty + verdicts** | `prereg_C_novelty_filter` | drop ChEMBL-approved + human-verified |

All biological features are **recomputed** per (drug, indication) pair from cached public biology —
never median-imputed (`no_impute_biological_features` rule). `prereg_C_build_genetics` de-imputes the
Mendelian/DepMap/mechanism-impact sub-blocks that were previously median-filled; a pair lacking targets
or an OT disease module gets a *computed* 0.0 ("no evidence"), not an imputed value. The only imputed
features left are non-biological trial-context interactions (leverage, trial-structure).

Heavy / non-deterministic stages (live ct.gov pulls, model fit) are pinned to their
**committed artifacts** (CSV + SHA-256). The deterministic filters are re-run live below.

In [1]:
import subprocess, hashlib, json
from pathlib import Path
import pandas as pd
ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
P = ROOT / "results/benchmark/prereg_C"
def run(script):
    r = subprocess.run(["python3", f"scripts/benchmark/{script}"], cwd=ROOT,
                       capture_output=True, text=True)
    print("\n".join(l for l in r.stdout.splitlines()
                     if "GPU Tree" not in l and "recompile" not in l))
    if r.returncode: print("STDERR:", r.stderr[-500:])
full = pd.read_csv(P / "prereg_C_locked_predictions.csv")
sha = hashlib.sha256((P / "prereg_C_locked_predictions.csv").read_bytes()).hexdigest()
print(f"Committed FULL lock: {len(full)} preds / {full.NCT_ID.nunique()} trials  (SHA {sha[:16]}…)")

Committed FULL lock: 711 preds / 469 trials  (SHA 47134a3c12dac64e…)


## Stage 3b — Biology recompute & honest feature-completeness

`prereg_C_build_genetics` recomputes the Mendelian/rare-variant causal, DepMap lineage-
dependency, and domain-conditional mechanism-impact features for every novel pair from cached
biology only (no live pulls). These were previously median-imputed; recomputing them means the
locked model sees **zero imputed biology** (`no_impute_biological_features`).

Two further one-time, auditable data steps remove the remaining gaps:
- `prereg_C_pull_missing_modules` pulls OT disease modules the cohort caches lacked — including the
  parent mapping *neurogenic detrusor overactivity → overactive bladder* (OT has no NDO entity;
  vibegron's target ADRB3 sits at ~0.61 in the OAB module, recovering the signal honestly).
- Drug targets missing from ChEMBL's curated table are web-verified and added to
  `ik14_moa_targets_combined_v1.csv` (e.g. hydroxychloroquine → TLR7/TLR9).

Completeness is then labelled three honest ways — **COMPLETE**, **N/A** (the drug acts without a
protein target: a DNA-damaging cytotoxic or a metabolic/cofactor MOA, so target→disease
mechanism-fit does not apply — e.g. melphalan, acetylcysteine, citrulline, thiamine), and
**INCOMPLETE** (a genuine resolvable data gap). "Targetless" is never asserted.

In [2]:
run("prereg_C_build_genetics.py")
gen = pd.read_csv(ROOT / "data/sources/mechanism_genetics_prereg_C.csv")
print(f"\nmech-genetics recomputed for {len(gen)} novel pairs | "
      f"Mendelian causal {(gen.mendel_max>0.5).mean():.0%} | "
      f"DepMap dependency {(gen._depmap_has==1).mean():.0%} | "
      f"any computed signal {(gen[['mendel_max','mi_genetics']].abs().sum(axis=1)>0).mean():.0%}")
# the locked set now carries NO imputed biology — only non-biological trial-context interactions remain
_b = pd.read_csv(P / "prereg_C_confident_bets.csv")
print("confident-bet feature_completeness (committed):")
print(_b.feature_completeness.value_counts().to_string())

wrote mechanism_genetics_prereg_C.csv: 412 novel pairs
  Mendelian causal evidence (mendel_max>0.5): 6.8%
  DepMap lineage dependency present (has=1): 26.9%
  routed to DepMap (oncology): 25.5%
  any computed genetics signal (mendel_max|mi_genetics>0): 26.7%

mech-genetics recomputed for 412 novel pairs | Mendelian causal 7% | DepMap dependency 27% | any computed signal 27%
confident-bet feature_completeness (committed):
feature_completeness
COMPLETE: all biology recomputed (mechanism + direct-target + Mendelian/DepMap/impact)                     33
N/A: drug acts without a protein target (cytotoxic/DNA or metabolic MOA) — mechanism-fit not applicable     1


## Stage 5 — Attribution: the drug must be the experimental *differentiator*

The earlier "clean" rule (named in brief title **and** in an experimental arm) admitted
**83/349 chemo/standard *backbones***. In *"novel agent + chemo  vs  placebo + chemo"*, the chemo
is named in the title and is in the experimental arm — but the PASS/FAIL is the **novel agent's**.
Correct test = **differentiator**: in an experimental arm but **not** the comparator arm.

In [3]:
cache = json.load(open(ROOT / "data/cache/ctgov_ongoing_p3.json"))
def arms(nct):
    for _d, ss in cache.items():
        for s in ss or []:
            p = s.get("protocolSection", {})
            if p.get("identificationModule", {}).get("nctId") == nct:
                e, c = [], []
                for a in p.get("armsInterventionsModule", {}).get("armGroups", []) or []:
                    (e if a.get("type") == "EXPERIMENTAL" else c).append(a.get("interventionNames"))
                return p["identificationModule"].get("briefTitle"), e, c
for nct in ["NCT05294172", "NCT06852222"]:
    t, e, c = arms(nct)
    print(nct, "—", t[:70]); print("   EXP :", e); print("   COMP:", c, "\n")

NCT05294172 — KL-A167 Injection Combined With Cisplatin and Gemcitabine vs Placebo C
   EXP : [['Drug: KL-A167', 'Drug: Gemcitabine', 'Drug: Cisplatin']]
   COMP: [['Drug: Gemcitabine', 'Drug: Cisplatin', 'Drug: Placebo']] 

NCT06852222 — A Study of Bleximenib, Venetoclax and Azacitidine For Treatment of Par
   EXP : [['Drug: Bleximenib', 'Drug: Venetoclax (VEN)', 'Drug: Azacitidine (AZA)']]
   COMP: [['Drug: Venetoclax (VEN)', 'Drug: Azacitidine (AZA)', 'Drug: Placebo']] 



In [4]:
run("prereg_C_finalize_clean.py")   # (fixed) attribution filter -> attribution-clean lock

CLEAN locked set: 111 predictions / 105 trials / 76 drugs (from full 711/469)
  predicted FAIL 9/111 (8%); median P_fail 0.168
  SHA-256 44fdab735fbeb9f1ff969a8be923e5e13c0a57603c04363106f73b6ce5d8f90e
  -> <repo>/results/benchmark/prereg_C/prereg_C_locked_predictions_clean.csv


## Stage 6 — Novelty (ChEMBL) + cytotoxic cap + committed human verdicts

1. **ChEMBL approval exclusion** (label-blind): drop pairs at ChEMBL `max_phase_for_ind == 4`.
2. **Cytotoxic-monotherapy cap**: oncology + cytotoxic-target + *monotherapy* (real ct.gov combo
   status) → cap `P_fail ≤ 0.15`.
3. **Committed human verdicts** (`data/sources/prereg_C_residual_verdicts_v1.csv`): the
   confident-FAIL residual is adjudicated by a web-verified determination; `EXCLUDE`
   (established/approved/SOC) or `KEEP-AMBIG` (vague indication) are dropped. Editing that CSV
   changes the final set deterministically.

In [5]:
run("prereg_C_novelty_filter.py")

deposited set feature completeness: 58 COMPLETE / 5 N/A (mechanism-fit not applicable) / 0 INCOMPLETE (resolvable data gap)
START (attribution-clean lock): 111 preds / 105 trials / 76 drugs (9 FAIL)
PASS 1 ChEMBL approved (phase 4) -> EXCLUDE 18 pairs (1 FAIL)
PASS 2 cytotoxic-mono cap: 1 pairs (real ct.gov combo status; NaN not capped: 0)
PASS 3 human verdicts on 8 confident-FAIL residual: EXCLUDE 1, KEEP-AMBIG 0, KEEP 7

FINAL locked set: 63 preds / 62 trials / 42 drugs (7 FAIL)
SHA-256 43923b0c8c37cd0a0991d6165e643aea36803e3323a0210dc90d9d386dcc286f
  -> prereg_C_locked_predictions_final.csv + residual audit prereg_C_residual_for_review.csv


In [6]:
final = pd.read_csv(P / "prereg_C_locked_predictions_final.csv")
sha = hashlib.sha256((P / "prereg_C_locked_predictions_final.csv").read_bytes()).hexdigest()
funnel = pd.DataFrame([
    ("0. full lock", 1470, 867),
    ("1. clean (old: title+exp)", 349, 292),
    ("2. + attribution (differentiator)", 265, 219),
    ("3. + ChEMBL-approved + cap", 222, 191),
    ("4. + human verdicts = FINAL", len(final), final.NCT_ID.nunique()),
], columns=["stage", "predictions", "trials"])
print(funnel.to_string(index=False))
print(f"\nFINAL: {len(final)} preds / {final.NCT_ID.nunique()} trials / {final.drug.nunique()} drugs "
      f"| predicted FAIL {(final.label_adj=='FAIL').sum()}")
print(f"SHA-256 {sha}")
final.sort_values('P_fail_overall_calibrated', ascending=False).head(8)

                            stage  predictions  trials
                     0. full lock         1470     867
        1. clean (old: title+exp)          349     292
2. + attribution (differentiator)          265     219
       3. + ChEMBL-approved + cap          222     191
      4. + human verdicts = FINAL           63      62

FINAL: 63 preds / 62 trials / 42 drugs | predicted FAIL 7
SHA-256 43923b0c8c37cd0a0991d6165e643aea36803e3323a0210dc90d9d386dcc286f


,NCT_ID,drug,novel_indication,indication_ctgov,feature_completeness,overall_status,P_fail_overall_raw,P_fail_overall_calibrated,P_fail_efficacy_calibrated,P_fail_safety_calibrated,model_prediction,P_fail_adj,label_adj,chembl_max_phase,cyto_mono,lock_date
0,NCT05066633,metoprolol,"Muscular Dystrophy, Duchenne","Muscular Dystrophy, Duchenne",COMPLETE: all biology recomputed (mechanism + ...,RECRUITING,0.9964,0.8848,0.6644,0.0538,FAIL_EFFICACY,0.8848,FAIL,NaN,False,2026-06-21
1,NCT04661358,fenofibrate,Diabetic Retinopathy,Diabetic Retinopathy,COMPLETE: all biology recomputed (mechanism + ...,RECRUITING,0.9960,0.8651,0.2984,0.4279,FAIL_SAFETY,0.8651,FAIL,3.0,False,2026-06-21
2,NCT05843643,upadacitinib,Systemic Lupus Erythematosus,Systemic Lupus Erythematosus,COMPLETE: all biology recomputed (mechanism + ...,ACTIVE_NOT_RECRUITING,0.9825,0.6178,0.7580,0.0353,FAIL_EFFICACY,0.6178,FAIL,2.0,False,2026-06-21
3,NCT06545526,metformin,Familial adenomatous polyposis,Familial Adenomatous Polyposis,COMPLETE: all biology recomputed (mechanism + ...,RECRUITING,0.9844,0.6108,0.5673,0.0329,FAIL_EFFICACY,0.6108,FAIL,2.0,False,2026-06-21
4,NCT06919835,cilostazol,Stroke,Stroke,COMPLETE: all biology recomputed (mechanism + ...,NOT_YET_RECRUITING,0.9815,0.6084,0.7138,0.0557,FAIL_EFFICACY,0.6084,FAIL,3.0,False,2026-06-21
5,NCT06987383,vibegron,obesity,Obesity,COMPLETE: all biology recomputed (mechanism + ...,NOT_YET_RECRUITING,0.9448,0.5061,0.3405,0.0674,FAIL_EFFICACY,0.5061,FAIL,NaN,False,2026-06-21
6,NCT06364904,trilaciclib,Bladder Cancer,Bladder Cancer,COMPLETE: all biology recomputed (mechanism + ...,NOT_YET_RECRUITING,0.9600,0.5054,0.2302,0.7628,FAIL_SAFETY,0.5054,FAIL,3.0,False,2026-06-21
7,NCT05195775,tadalafil,Duchenne Muscular Dystrophy,Duchenne Muscular Dystrophy,COMPLETE: all biology recomputed (mechanism + ...,ACTIVE_NOT_RECRUITING,0.9347,0.4644,0.5758,0.0175,PASS,0.4644,PASS,3.0,False,2026-06-21


## Reproduce / caveats

```bash
# biology recompute (deterministic, cache-only — no live pulls):
python3 scripts/benchmark/prereg_C_build_mech.py        # topology/KEGG/OT/ClinGen/in-module + direct-target
python3 scripts/benchmark/prereg_C_build_genetics.py    # Mendelian/DepMap/mechanism-impact (de-imputed)
python3 scripts/benchmark/prereg_C_pull_missing_modules.py  # OT disease modules absent from cohort caches (idempotent)
# then the lock consumes both (live ct.gov pull + model fit; committed artifact is the registration):
python3 scripts/benchmark/prereg_C_lock.py
# deterministic downstream filters:
python3 scripts/benchmark/prereg_C_finalize_clean.py   # attribution-clean lock
python3 scripts/benchmark/prereg_C_novelty_filter.py   # novelty + verdicts -> FINAL + SHA
python3 scripts/benchmark/prereg_C_confidence_gate.py  # selective-prediction gate -> confident bets
```

- `prereg_C_build_mech.py` and `prereg_C_build_genetics.py` are **deterministic and cache-only**
  (ClinVar / Open-Targets-causal / DepMap CRISPR / OmniPath / KEGG) — they recompute the full
  biological feature set per novel pair with no live pulls, so the model never sees imputed biology.
- `prereg_C_pull_missing_modules.py` is a one-time OT pull (idempotent; committed caches already hold
  the modules, so it is a no-op offline). Drug-target curation gaps (e.g. hydroxychloroquine →
  TLR7/TLR9) are web-verified additions to `ik14_moa_targets_combined_v1.csv`.
- Feature-completeness is labelled COMPLETE / N/A (no protein target — mechanism-fit N/A) / INCOMPLETE
  (resolvable gap) by `prereg_C_completeness.py`; the deposited set is 0 INCOMPLETE.
- The **full lock** (`prereg_C_lock.py`) does a *live* ct.gov pull + model fit; its CSV + SHA are
  committed and treated as the fixed input (re-running may drift if ct.gov changes — the committed
  artifact is the registration).
- Both filters above are **deterministic** from committed inputs, so FINAL + SHA reproduce exactly.

## Stage 7 — Confidence gate (selective prediction)

Not all forward predictions are equal. We LOCK a prediction only where the model is
demonstrably reliable on held-out data, and ABSTAIN on the uncertain middle. Two thresholds
are *derived* from the canonical out-of-fold OVERALL predictions: the widest low-P_fail band
with held-out PASS precision ≥ 90%, and the narrowest high-P_fail band with held-out FAIL
precision ≥ 90%. The thresholds are set on held-out **cohort** indications; the forward pairs
are **novel** (out-of-distribution), so whether the reliability transfers is itself part of
what the prospective test measures — disclosed, not assumed.

In [7]:
run("prereg_C_confidence_gate.py")

Held-out OOF (2202 rows): bands for >= 90% precision (min n=20)
  confident PASS: P_fail <= 0.21  precision 0.905  (n=1213, 55% coverage)
  confident FAIL: P_fail >= 0.54  precision 0.900  (n=60, 3% coverage)
  feature completeness: 58 COMPLETE / 5 N/A (mechanism-fit not applicable) / 0 INCOMPLETE (resolvable data gap)

OOD/coverage handling (confident-FAIL only):
  ABSTAIN 0 confident-FAIL with mechanism coverage ABSENT (structural-zero, not evidence)
  FLAG raw_extrapolation (raw P_fail > OOF 95th pct 0.918): disclosed, NOT abstained

Confidence gate (confident-FAIL only; support<10 trials AND no sub-head >= 0.5):
  ABSTAIN 1 confident-FAIL as LOW-confidence (thin disease support AND uncorroborated)
    fenofibrate        -> Diabetic Retinopathy                       (n_dis=1, subhead_max=0.43, conf=0.41)

FINAL set 63 -> CONFIDENT BETS 34 (54%): 30 PASS-bets, 4 FAIL-bets | ABSTAIN 29
expected accuracy at readout (held-out): PASS-bets ~91%, FAIL-bets ~90%
SHA-256 bd2ce3939056babf725f

In [8]:
bets = pd.read_csv(P / "prereg_C_confident_bets.csv")
sha = hashlib.sha256((P / "prereg_C_confident_bets.csv").read_bytes()).hexdigest()
print(f"REGISTERED confident bets: {len(bets)}  ({(bets.bet=='PASS').sum()} PASS, "
      f"{(bets.bet=='FAIL').sum()} FAIL)   SHA-256 {sha[:16]}…")
bets[bets.bet=='FAIL'][['drug','novel_indication','P_fail_overall_calibrated','model_prediction']]

REGISTERED confident bets: 34  (30 PASS, 4 FAIL)   SHA-256 bd2ce3939056babf…


,drug,novel_indication,P_fail_overall_calibrated,model_prediction
0,metoprolol,"Muscular Dystrophy, Duchenne",0.8848,FAIL_EFFICACY
1,upadacitinib,Systemic Lupus Erythematosus,0.6178,FAIL_EFFICACY
2,metformin,Familial adenomatous polyposis,0.6108,FAIL_EFFICACY
3,cilostazol,Stroke,0.6084,FAIL_EFFICACY


In [9]:
# Human-signoff sheet: join the committed verdicts to the CURRENT post-gate live set so the
# reviewer only spends attention on verdicts that still affect a confident bet.
run("prereg_C_signoff_sheet.py")

wrote results/benchmark/prereg_C/prereg_C_signoff_current.xlsx

78 committed verdicts. Review burden by priority:
review_priority
1-REVIEW (live FAIL bet)            4
2-REVIEW (ambiguous indication)     3
4-no action (not live)             71

=== rows that actually need your eyes (priority 1–2) ===
  [1-REVIEW (live FAIL bet)] cilostazol -> Stroke  (KEEP/high; LIVE — confident FAIL bet)
  [1-REVIEW (live FAIL bet)] metformin -> Familial adenomatous polyposis  (KEEP/high; LIVE — confident FAIL bet)
  [1-REVIEW (live FAIL bet)] metoprolol -> Muscular Dystrophy, Duchenne  (KEEP/med; LIVE — confident FAIL bet)
  [1-REVIEW (live FAIL bet)] upadacitinib -> Systemic Lupus Erythematosus  (KEEP/high; LIVE — confident FAIL bet)
  [2-REVIEW (ambiguous indication)] alectinib -> Cancer  (KEEP-AMBIG/low; filtered out of funnel (not a current bet))
  [2-REVIEW (ambiguous indication)] sunitinib -> Metastatic Cancer  (KEEP-AMBIG/low; filtered out of funnel (not a current bet))
  [2-REVIEW (ambiguous 

In [10]:
# Reviewer sheet for the 81 registered bets: NCT links + per-bet model rationale grounded in the
# committed per-pair mechanism feature values (the signal that differentiates a novel-pair FAIL from PASS).
run("prereg_C_bets_for_review.py")

wrote results/benchmark/prereg_C/prereg_C_bets_for_review.xlsx (0 INCOMPLETE rows highlighted)
34 bets: 4 FAIL (review), 30 PASS

=== 10 FAIL bets: NCT + rationale ===

  metoprolol -> Muscular Dystrophy, Duchenne  (P_fail 0.88)
    https://clinicaltrials.gov/study/NCT05066633
    EFFICACY-driven mechanism mismatch: the target is NOT a genetically-supported driver of this indication (OT genetic ≈ 0 and target absent from the disease gene module) — the drug can be right, the disease wrong. Target is network-distal/upstream from the disease module (negative topology).

  upadacitinib -> Systemic Lupus Erythematosus  (P_fail 0.62)
    https://clinicaltrials.gov/study/NCT05843643
    EFFICACY-driven mechanism mismatch: the target is NOT a genetically-supported driver of this indication (OT genetic ≈ 0 and target absent from the disease gene module) — the drug can be right, the disease wrong.

  metformin -> Familial adenomatous polyposis  (P_fail 0.61)
    https://clinicaltrials.gov/study/